# Evaluasi & Inference - Plate Count Reader

Notebook ini melakukan evaluasi komprehensif terhadap model multi-class
**Plate Count Reader** dan mendemonstrasikan inference pada gambar baru.

## Parameter (sesuai train_18k_gpu1.py)
- **Model path**: `runs/18k_multiclass_gpu1/weights/best.pt`
- **Dataset**: `data/yolo_18k_multiclass/`
- **CONF_THRESHOLD**: 0.20
- **Kelas**: colony (0), bubble (1), dust (2), crack (3)

## Setup Environment

In [ ]:
# Import libraries
import os
import glob
import json
import random
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

from ultralytics import YOLO

# Configuration (SESUAI train_18k_gpu1.py)
WORKSPACE = Path('/media/lambda_one/DFSSD04/project/healtcare')
YOLO_DIR = WORKSPACE / 'data' / 'yolo_18k_multiclass'  # FIXED: was yolo_multiclass
DATA_YAML = YOLO_DIR / 'data.yaml'

# Model paths (SESUAI train_18k_gpu1.py)
RUNS_DIR = WORKSPACE / 'runs'
RUN_NAME = '18k_multiclass_gpu1'
WEIGHTS_DIR = RUNS_DIR / RUN_NAME / 'weights'
MULTICLASS_MODEL = WEIGHTS_DIR / 'best.pt'  # Load dari actual training output
FALLBACK_MODEL = WORKSPACE / 'models' / 'best_multiclass_18k.pt'

# Class config
CLASS_NAMES = ['colony', 'bubble', 'dust', 'crack']
NUM_CLASSES = len(CLASS_NAMES)
CLASS_COLORS = {
    0: (0, 255, 0),     # colony - green
    1: (0, 165, 255),   # bubble - orange
    2: (128, 128, 128), # dust - gray
    3: (0, 0, 255),     # crack - red
}

DEVICE = '1'
IMG_SIZE = 640
CONF_THRESHOLD = 0.20  # FIXED: was 0.25, SESUAI train_18k_gpu1.py PL_CONF
IOU_THRESHOLD = 0.45

random.seed(42)
np.random.seed(42)

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print('Environment ready')

## Load Model & Dataset

In [ ]:
# Load model multi-class (prioritas: WEIGHTS_DIR/best.pt > fallback)
model_path = MULTICLASS_MODEL if MULTICLASS_MODEL.exists() else FALLBACK_MODEL
assert model_path.exists(), f'Model tidak ditemukan: {model_path} (juga cek {FALLBACK_MODEL})'

model = YOLO(str(model_path))
print(f'Model loaded: {model_path}')
print(f'   Classes: {model.names}')
print(f'   Model size: {model_path.stat().st_size / 1024 / 1024:.2f} MB')

In [ ]:
# Verifikasi test dataset (SESUAI train_18k_gpu1.py structure: {split}/images/)
test_img_dir = YOLO_DIR / 'test' / 'images'
test_lbl_dir = YOLO_DIR / 'test' / 'labels'

test_images = sorted(list(test_img_dir.glob('*.jpg')) + list(test_img_dir.glob('*.png')))
test_labels = sorted(list(test_lbl_dir.glob('*.txt')))

print(f'Test Dataset:')
print(f'   Images: {len(test_images)}')
print(f'   Labels: {len(test_labels)}')
print(f'   Match : {"OK" if len(test_images) == len(test_labels) else "MISMATCH!"}')

## Metrik Evaluasi Keseluruhan

In [ ]:
# Evaluasi pada test set
results = model.val(
    data=str(DATA_YAML),
    split='test',
    device=DEVICE,
    imgsz=IMG_SIZE,
    conf=CONF_THRESHOLD,
    iou=IOU_THRESHOLD,
    verbose=True,
    plots=True,
    save_json=True,
)

print('\n' + '=' * 50)
print('METRIK EVALUASI KESELURUHAN')
print('=' * 50)
print(f'  mAP50      : {results.box.map50:.4f}')
print(f'  mAP50-95   : {results.box.map:.4f}')
print(f'  Precision   : {results.box.mp:.4f}')
print(f'  Recall      : {results.box.mr:.4f}')
f1 = 2 * results.box.mp * results.box.mr / (results.box.mp + results.box.mr + 1e-8)
print(f'  F1 Score    : {f1:.4f}')
print('=' * 50)

## Metrik Per Kelas

In [ ]:
# Per-class metrics
class_maps50 = results.box.maps50 if hasattr(results.box, 'maps50') else results.box.ap50
class_maps = results.box.maps if hasattr(results.box, 'maps') else results.box.ap

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# mAP50 per class
bars = axes[0].bar(CLASS_NAMES, class_maps50, color=['#4CAF50', '#FF9800', '#9E9E9E', '#F44336'], alpha=0.8)
axes[0].set_title('mAP50 Per Kelas', fontweight='bold')
axes[0].set_ylabel('mAP50')
axes[0].set_ylim(0, 1.0)
for bar, val in zip(bars, class_maps50):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=10)

# mAP50-95 per class
bars = axes[1].bar(CLASS_NAMES, class_maps, color=['#4CAF50', '#FF9800', '#9E9E9E', '#F44336'], alpha=0.8)
axes[1].set_title('mAP50-95 Per Kelas', fontweight='bold')
axes[1].set_ylabel('mAP50-95')
axes[1].set_ylim(0, 1.0)
for bar, val in zip(bars, class_maps):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=10)

# Comparison
x = np.arange(NUM_CLASSES)
width = 0.35
axes[2].bar(x - width/2, class_maps50, width, label='mAP50', color='#2196F3', alpha=0.8)
axes[2].bar(x + width/2, class_maps, width, label='mAP50-95', color='#FF5722', alpha=0.8)
axes[2].set_title('Perbandingan mAP50 vs mAP50-95', fontweight='bold')
axes[2].set_ylabel('Score')
axes[2].set_xticks(x)
axes[2].set_xticklabels(CLASS_NAMES)
axes[2].legend()
axes[2].set_ylim(0, 1.0)

plt.tight_layout()
plt.show()

print('\nMetrik Detail Per Kelas:')
print(f'{"Kelas":<10} {"mAP50":<12} {"mAP50-95":<12} {"Gap":<10}')
print('-' * 44)
for i, name in enumerate(CLASS_NAMES):
    gap = class_maps50[i] - class_maps[i]
    print(f'{name:<10} {class_maps50[i]:<12.4f} {class_maps[i]:<12.4f} {gap:<10.4f}')

## Confusion Matrix Analysis

In [ ]:
# Display confusion matrix
from IPython.display import Image as IPImage, display

cm_dir = Path(results.save_dir) if hasattr(results, 'save_dir') else RUNS_DIR / RUN_NAME
cm_paths = list(cm_dir.rglob('confusion_matrix*.png'))

for cm_path in cm_paths:
    print(f'{cm_path.name}:')
    display(IPImage(filename=str(cm_path)))

if not cm_paths:
    print('Confusion matrix plot tidak ditemukan dari results.')

In [ ]:
# Hitung confusion matrix manual dari test predictions
confusion_data = np.zeros((NUM_CLASSES + 1, NUM_CLASSES + 1), dtype=int)

for img_path in test_images:
    lbl_path = test_lbl_dir / (img_path.stem + '.txt')
    
    gt_boxes = []
    if lbl_path.exists():
        with open(str(lbl_path), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    gt_boxes.append({
                        'class': int(parts[0]),
                        'cx': float(parts[1]), 'cy': float(parts[2]),
                        'w': float(parts[3]), 'h': float(parts[4])
                    })
    
    preds = model.predict(str(img_path), device=DEVICE, imgsz=IMG_SIZE, 
                          conf=CONF_THRESHOLD, iou=IOU_THRESHOLD, verbose=False)
    
    pred_boxes = []
    if preds and len(preds) > 0 and preds[0].boxes is not None:
        for box in preds[0].boxes:
            xywhn = box.xywhn[0].cpu().numpy()
            pred_boxes.append({
                'class': int(box.cls[0].cpu().numpy()),
                'cx': float(xywhn[0]), 'cy': float(xywhn[1]),
                'w': float(xywhn[2]), 'h': float(xywhn[3]),
                'conf': float(box.conf[0].cpu().numpy())
            })
    
    matched_gt = set()
    matched_pred = set()
    
    for pi, pred in enumerate(pred_boxes):
        best_iou = 0
        best_gi = -1
        for gi, gt in enumerate(gt_boxes):
            if gi in matched_gt:
                continue
            ix1 = max(pred['cx'] - pred['w']/2, gt['cx'] - gt['w']/2)
            iy1 = max(pred['cy'] - pred['h']/2, gt['cy'] - gt['h']/2)
            ix2 = min(pred['cx'] + pred['w']/2, gt['cx'] + gt['w']/2)
            iy2 = min(pred['cy'] + pred['h']/2, gt['cy'] + gt['h']/2)
            iw = max(0, ix2 - ix1)
            ih = max(0, iy2 - iy1)
            inter = iw * ih
            union = pred['w'] * pred['h'] + gt['w'] * gt['h'] - inter
            if union > 0:
                iou = inter / union
                if iou > best_iou and iou > 0.3:
                    best_iou = iou
                    best_gi = gi
        
        if best_gi >= 0:
            confusion_data[gt_boxes[best_gi]['class'], pred['class']] += 1
            matched_gt.add(best_gi)
            matched_pred.add(pi)
        else:
            confusion_data[NUM_CLASSES, pred['class']] += 1
    
    for gi, gt in enumerate(gt_boxes):
        if gi not in matched_gt:
            confusion_data[gt['class'], NUM_CLASSES] += 1

# Visualize
fig, ax = plt.subplots(figsize=(8, 8))
labels = CLASS_NAMES + ['background']
im = ax.imshow(confusion_data, cmap='Blues', interpolation='nearest')
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels)
ax.set_xlabel('Predicted')
ax.set_ylabel('Ground Truth')
ax.set_title('Confusion Matrix (Manual)', fontweight='bold')

for i in range(len(labels)):
    for j in range(len(labels)):
        text_color = 'white' if confusion_data[i, j] > confusion_data.max() / 2 else 'black'
        ax.text(j, i, str(confusion_data[i, j]), ha='center', va='center', 
                color=text_color, fontsize=11)

plt.colorbar(im)
plt.tight_layout()
plt.show()

## Visualisasi Prediksi

In [ ]:
# Visualisasi prediksi pada gambar test
def draw_predictions(img_path, model, class_names=CLASS_NAMES, class_colors=CLASS_COLORS,
                     conf_threshold=CONF_THRESHOLD, save_path=None):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    preds = model.predict(str(img_path), device=DEVICE, imgsz=IMG_SIZE,
                          conf=conf_threshold, verbose=False)
    
    if preds and len(preds) > 0 and preds[0].boxes is not None:
        for box in preds[0].boxes:
            cls_id = int(box.cls[0].cpu().numpy())
            conf = float(box.conf[0].cpu().numpy())
            xyxy = box.xyxy[0].cpu().numpy()
            x1, y1, x2, y2 = map(int, xyxy)
            color = class_colors.get(cls_id, (255, 255, 255))
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            label = f'{class_names[cls_id]} {conf:.2f}'
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(img, (x1, y1 - th - 10), (x1 + tw + 10, y1), color, -1)
            cv2.putText(img, label, (x1 + 5, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
    return img

sample_imgs = random.sample(test_images, min(8, len(test_images)))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for idx, img_path in enumerate(sample_imgs):
    row, col = idx // 4, idx % 4
    annotated = draw_predictions(img_path, model)
    axes[row, col].imshow(annotated)
    axes[row, col].axis('off')
    axes[row, col].set_title(img_path.stem[:25], fontsize=9)

plt.suptitle('Prediksi Model pada Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Analisis Error

In [ ]:
# Analisis False Positives dan False Negatives
error_analysis = {
    'fp_per_class': Counter(),
    'fn_per_class': Counter(),
    'confusion_pairs': Counter(),
}

for img_path in test_images:
    lbl_path = test_lbl_dir / (img_path.stem + '.txt')
    
    gt_boxes = []
    if lbl_path.exists():
        with open(str(lbl_path), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    gt_boxes.append({
                        'class': int(parts[0]),
                        'cx': float(parts[1]), 'cy': float(parts[2]),
                        'w': float(parts[3]), 'h': float(parts[4])
                    })
    
    preds = model.predict(str(img_path), device=DEVICE, imgsz=IMG_SIZE,
                          conf=CONF_THRESHOLD, verbose=False)
    pred_boxes = []
    if preds and len(preds) > 0 and preds[0].boxes is not None:
        for box in preds[0].boxes:
            xywhn = box.xywhn[0].cpu().numpy()
            pred_boxes.append({
                'class': int(box.cls[0].cpu().numpy()),
                'cx': float(xywhn[0]), 'cy': float(xywhn[1]),
                'w': float(xywhn[2]), 'h': float(xywhn[3]),
                'conf': float(box.conf[0].cpu().numpy())
            })
    
    matched_gt = set()
    for pred in pred_boxes:
        best_iou = 0
        best_gi = -1
        for gi, gt in enumerate(gt_boxes):
            if gi in matched_gt:
                continue
            ix1 = max(pred['cx'] - pred['w']/2, gt['cx'] - gt['w']/2)
            iy1 = max(pred['cy'] - pred['h']/2, gt['cy'] - gt['h']/2)
            ix2 = min(pred['cx'] + pred['w']/2, gt['cx'] + gt['w']/2)
            iy2 = min(pred['cy'] + pred['h']/2, gt['cy'] + gt['h']/2)
            iw = max(0, ix2 - ix1)
            ih = max(0, iy2 - iy1)
            inter = iw * ih
            union = pred['w'] * pred['h'] + gt['w'] * gt['h'] - inter
            if union > 0:
                iou = inter / union
                if iou > best_iou and iou > 0.3:
                    best_iou = iou
                    best_gi = gi
        
        if best_gi >= 0:
            gt_cls = gt_boxes[best_gi]['class']
            pred_cls = pred['class']
            if gt_cls != pred_cls:
                error_analysis['confusion_pairs'][(CLASS_NAMES[gt_cls], CLASS_NAMES[pred_cls])] += 1
            matched_gt.add(best_gi)
        else:
            error_analysis['fp_per_class'][CLASS_NAMES[pred['class']]] += 1
    
    for gi, gt in enumerate(gt_boxes):
        if gi not in matched_gt:
            error_analysis['fn_per_class'][CLASS_NAMES[gt['class']]] += 1

# Visualisasi
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

fp_vals = [error_analysis['fp_per_class'].get(CLASS_NAMES[i], 0) for i in range(NUM_CLASSES)]
fn_vals = [error_analysis['fn_per_class'].get(CLASS_NAMES[i], 0) for i in range(NUM_CLASSES)]

bars = ax1.bar(CLASS_NAMES, fp_vals, color=['#4CAF50', '#FF9800', '#9E9E9E', '#F44336'], alpha=0.8)
ax1.set_title('False Positives Per Kelas', fontweight='bold')
for bar, val in zip(bars, fp_vals):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1, str(val), ha='center')

bars = ax2.bar(CLASS_NAMES, fn_vals, color=['#4CAF50', '#FF9800', '#9E9E9E', '#F44336'], alpha=0.8)
ax2.set_title('False Negatives Per Kelas', fontweight='bold')
for bar, val in zip(bars, fn_vals):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1, str(val), ha='center')

plt.tight_layout()
plt.show()

if error_analysis['confusion_pairs']:
    print('\nTop Confusion Pairs (GT -> Predicted):')
    for (gt_cls, pred_cls), count in error_analysis['confusion_pairs'].most_common(10):
        print(f'   {gt_cls} -> {pred_cls}: {count}')

## Inference pada Gambar Baru

In [ ]:
# Fungsi inference untuk gambar baru
def predict_and_visualize(image_path, model, save_output=True, output_dir=None):
    img = cv2.imread(str(image_path))
    if img is None:
        print(f'Gagal membaca gambar: {image_path}')
        return None, []
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    preds = model.predict(str(image_path), device=DEVICE, imgsz=IMG_SIZE,
                          conf=CONF_THRESHOLD, iou=IOU_THRESHOLD, verbose=False)
    
    detections = []
    if preds and len(preds) > 0 and preds[0].boxes is not None:
        for box in preds[0].boxes:
            cls_id = int(box.cls[0].cpu().numpy())
            conf = float(box.conf[0].cpu().numpy())
            xyxy = box.xyxy[0].cpu().numpy()
            
            detections.append({
                'class': CLASS_NAMES[cls_id],
                'class_id': cls_id,
                'confidence': conf,
                'bbox': xyxy.tolist()
            })
            
            x1, y1, x2, y2 = map(int, xyxy)
            color = CLASS_COLORS.get(cls_id, (255, 255, 255))
            cv2.rectangle(img_rgb, (x1, y1), (x2, y2), color, 3)
            label = f'{CLASS_NAMES[cls_id]} {conf:.2f}'
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(img_rgb, (x1, y1 - th - 12), (x1 + tw + 12, y1), color, -1)
            cv2.putText(img_rgb, label, (x1 + 6, y1 - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    colony_count = sum(1 for d in detections if d['class'] == 'colony')
    cv2.putText(img_rgb, f'Colony Count: {colony_count}', (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
    
    if save_output and output_dir:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        out_path = output_dir / f'pred_{Path(image_path).stem}.jpg'
        cv2.imwrite(str(out_path), cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
    
    return img_rgb, detections

# Test pada sample dari test set
if test_images:
    sample_path = test_images[0]
    pred_img, detections = predict_and_visualize(sample_path, model)
    
    plt.figure(figsize=(12, 8))
    plt.imshow(pred_img)
    plt.axis('off')
    plt.title('Inference Sample', fontweight='bold')
    plt.show()
    
    print(f'\nDetections ({len(detections)} total):')
    for det in detections:
        print(f'   {det["class"]:<10} conf={det["confidence"]:.3f}')

## Export untuk Produksi

In [ ]:
# Export model ke berbagai format
export_dir = WORKSPACE / 'exports'
export_dir.mkdir(exist_ok=True)

if model_path.exists():
    export_model = YOLO(str(model_path))
    exports = {}
    
    # ONNX
    try:
        onnx_path = export_model.export(format='onnx', imgsz=IMG_SIZE, simplify=True, device=DEVICE)
        exports['ONNX'] = onnx_path
        print(f'ONNX: {onnx_path}')
    except Exception as e:
        print(f'ONNX export failed: {e}')
    
    # TorchScript
    try:
        ts_path = export_model.export(format='torchscript', imgsz=IMG_SIZE, device=DEVICE)
        exports['TorchScript'] = ts_path
        print(f'TorchScript: {ts_path}')
    except Exception as e:
        print(f'TorchScript export failed: {e}')
    
    print(f'\nTotal exports: {len(exports)}')
else:
    print('Model tidak tersedia untuk export')

## Ringkasan & Rekomendasi

In [ ]:
# Ringkasan final
print('=' * 70)
print('RINGKASAN EVALUASI MODEL PLATE COUNT READER')
print('=' * 70)
print()
print(f'{"Parameter":<35} {"Nilai"}')
print('-' * 70)
print(f'{"Model":<35} YOLOv8s Multi-Class')
print(f'{"Kelas":<35} {CLASS_NAMES}')
print(f'{"Model Path":<35} {model_path}')
print(f'{"CONF_THRESHOLD":<35} {CONF_THRESHOLD}')
print()

if 'results' in dir():
    print('Metrik Evaluasi Keseluruhan (Test Set):')
    print(f'  mAP50      : {results.box.map50:.4f}')
    print(f'  mAP50-95   : {results.box.map:.4f}')
    print(f'  Precision   : {results.box.mp:.4f}')
    print(f'  Recall      : {results.box.mr:.4f}')
    f1 = 2 * results.box.mp * results.box.mr / (results.box.mp + results.box.mr + 1e-8)
    print(f'  F1 Score    : {f1:.4f}')
    print()
    
    print('Metrik Per Kelas:')
    for i, name in enumerate(CLASS_NAMES):
        print(f'  {name:<10}: mAP50={class_maps50[i]:.4f}, mAP50-95={class_maps[i]:.4f}')

print()
total_fp = sum(error_analysis['fp_per_class'].values())
total_fn = sum(error_analysis['fn_per_class'].values())
print(f'Analisis Error:')
print(f'  Total False Positives : {total_fp}')
print(f'  Total False Negatives : {total_fn}')

print()
print('Rekomendasi:')
print('  1. Jika mAP colony rendah: tambah pseudo-label, tingkatkan imgsz')
print('  2. Jika bubble/dust/crack banyak FP: tambah data sintetis realistis')
print('  3. Untuk produksi: gunakan ONNX untuk API server, TFLite untuk mobile')
print('  4. Iterasi berikutnya: coba YOLOv8m/l, augmentation lebih agresif')
print('=' * 70)